# Extract Residual Stream Activations

This notebook demonstrates how to capture residual-stream activations during
vLLM inference using the async engine and vllm-lens.

## Imports

In [1]:
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

## Configuration

In [2]:
MODEL = "facebook/opt-125m"
PROMPT = "The future of AI is"
LAYERS = [0, 2, 5]

## Initialize the Async Engine

In [ ]:
engine_args = AsyncEngineArgs(
    model=MODEL,
    gpu_memory_utilization=0.85,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

## Generate with Activation Capture

Pass `output_residual_stream` via `extra_args` to request residual-stream
activations at the specified layers.

In [4]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=64,
    extra_args={"output_residual_stream": LAYERS},
)

final = None
async for output in engine.generate(PROMPT, sampling_params, "req-0"):
    final = output

## Inspect Results

In [5]:
print(f"Prompt: {PROMPT}")
print(f"Generated: {final.outputs[0].text}")

activation_data = getattr(final, "activations", None)
if activation_data is not None:
    residual_stream = activation_data["residual_stream"]
    print(f"\nResidual stream shape: {residual_stream.shape}")
    print(f"Sample activations (layer 0, first token): {residual_stream[0, 0, :5]}")
else:
    print("Activations: NOT FOUND (ERROR)")

Prompt: The future of AI is
Generated:  cloud computing

Jose Antonia

AI is destined to become a future of computing. The future of computing will be cloud computing, where it is no longer an urban fantasy.

The future of computing will be cloud computing, where it is no longer an urban fantasy. The future of computing will be cloud

Residual stream shape: torch.Size([3, 69, 768])
Sample activations (layer 0, first token): tensor([-0.1010, -1.0039,  0.1498,  0.1503,  0.0785], dtype=torch.float16)
